In [1]:
# Steps 2-4: preprocess every image, extract features, cache to feat/*.npy.
# Run this notebook ONCE; the model notebook then loads the .npy arrays directly.
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

# --- locate project root (nearest ancestor of CWD containing a 'data' folder) ---
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "data")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    raise FileNotFoundError("could not find project root containing a 'data' folder")

PATH = find_root()
DATA = os.path.join(PATH, "data")
FEAT = os.path.join(PATH, "feat")
os.makedirs(FEAT, exist_ok=True)
print("project root:", PATH)

RESIZE = (128, 64)  # (height, width); clean multiples of 8 for 8x8 HOG cells

train_df = pd.read_csv(os.path.join(DATA, "train.csv"))   # id, label
test_df  = pd.read_csv(os.path.join(DATA, "test.csv"))    # id

project root: c:\Users\dariabulacu\Documents\Facultate II-2\IA\ML\ML_Project


In [2]:
# Step 2 - preprocessing (single source of truth for every feature extractor)
from skimage.transform import resize

# transforms the image from RGBA -> Grayscale for the HOG processing
def load_gray(path):
    img = Image.open(path).convert("L")              # drops alpha + color -> luminance
    arr = np.asarray(img, dtype=np.float32) / 255.0  # normalize to [0,1]
    arr = resize(arr, RESIZE, anti_aliasing=True)    # uniform clean size (128x64)
    return arr

In [3]:
from skimage.feature import hog

def hog_feature(gray):
    return hog( gray,
                orientations=9, #orientation ranges from 0-180 degrees, dividing into 20 degrees histogram bins
                pixels_per_cell=(8,8),
                cells_per_block=(2,2),
                block_norm="L2-Hys",
                feature_vector=True)

In [4]:
# Step 4 - build feature matrix (row i corresponds to df.id.iloc[i])
def build_matrix(df, split):
    """Returns X_hog (N, ~3780): the HOG descriptor of every image."""
    X_hog = []
    for fn in tqdm(df["id"], desc=f"features [{split}]"):
        p = os.path.join(DATA, split, fn)
        g = load_gray(p)
        X_hog.append(hog_feature(g))
    return np.asarray(X_hog, np.float32)

In [5]:
# TRAIN: features + labels (skip if already cached; delete feat/*.npy to rebuild)
if not os.path.exists(os.path.join(FEAT, "Xtr_hog.npy")):
    Xtr_hog = build_matrix(train_df, "train")
    np.save(os.path.join(FEAT, "Xtr_hog.npy"),   Xtr_hog)
    np.save(os.path.join(FEAT, "y.npy"),         train_df["label"].to_numpy())
    print("train features:", Xtr_hog.shape)
else:
    print("train features already cached")

train features already cached


In [6]:
# TEST: features + ids in matching order (NO labels). test_ids lets the
# submission pair each prediction with the right filename.
if not os.path.exists(os.path.join(FEAT, "Xte_hog.npy")):
    Xte_hog = build_matrix(test_df, "test")
    np.save(os.path.join(FEAT, "Xte_hog.npy"),   Xte_hog)
    np.save(os.path.join(FEAT, "test_ids.npy"),  test_df["id"].to_numpy())
    print("test features:", Xte_hog.shape)
else:
    print("test features already cached ")

test features already cached 
